In [3]:
from ultralytics import YOLO, RTDETR

import cv2
import numpy as np
import pandas as pd
import torch
import time

from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

In [4]:
SEQ_PATH = Path(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\VisDrone2019-MOT-val\VisDrone2019-MOT-val\sequences\uav0000086_00000_v"
)

frames = sorted(
    list(SEQ_PATH.glob("*.jpg"))
)

print("Frames:",len(frames))


sample=cv2.imread(str(frames[0]))

H,W,_ = sample.shape

print(W,H)

Frames: 464
1344 756


In [5]:
experiments = [

    {
        "name":"YOLO11s_1280",
        "model":YOLO("yolo11s.pt"),
        "imgsz":1280
    },

    {
        "name":"YOLO11l_1280",
        "model":YOLO("yolo11l.pt"),
        "imgsz":1280
    },

    {
        "name":"RTDETR_960",
        "model":RTDETR("rtdetr-l.pt"),
        "imgsz":960
    }
]


for exp in experiments:
    
    exp["model"].to("cuda")

In [6]:
!pip install ultralytics motmetrics lap

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ----------------------------------- ---- 1.3/1.5 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.0 MB/s  0:00:00

   ---------------------------------------- 3/3 [motmetrics]



In [ ]:
from ultralytics import YOLO, RTDETR

from pathlib import Path
from tqdm import tqdm

import cv2
import torch
import numpy as np
import pandas as pd
import time

# import motmetrics as mm

In [15]:
import numpy as np

# patch for motmetrics + numpy 2.x compatibility
if not hasattr(np, "asfarray"):
    np.asfarray = lambda x, dtype=None: np.asarray(x, dtype=dtype)

In [16]:
import motmetrics as mm

In [54]:
VAL_ROOT = Path(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\VisDrone2019-MOT-val\VisDrone2019-MOT-val"
)

SEQ_DIR = VAL_ROOT / "sequences"

ANN_DIR = VAL_ROOT / "annotations"


sequences = sorted(
    list(SEQ_DIR.iterdir())
)[:2]


print(
    "Testing sequences:"
)

for s in sequences:
    print(s.name)

Testing sequences:
uav0000086_00000_v
uav0000117_02622_v


In [55]:
experiments = [

{
"name":"YOLO11s_1280",
"model":YOLO("yolo11s.pt"),
"imgsz":1280
},

{
"name":"YOLO11m_1280",
"model":YOLO("yolo11m.pt"),
"imgsz":1280
},


{
"name":"YOLO11l_1280",
"model":YOLO("yolo11l.pt"),
"imgsz":1280
},


{
"name":"RTDETR_960",
"model":RTDETR("rtdetr-l.pt"),
"imgsz":960
}

]


for e in experiments:

    e["model"].to("cuda")

In [57]:
trackers = {

    "ByteTrack":
    "bytetrack.yaml",




    "BoT-SORT-No-GMC":
    "Trackers/botsort_no_gmc_drone.yaml",


    "BoT-SORT-GMC":
    "Trackers/botsort_gmc_drone.yaml",

}

In [58]:
def load_gt(annotation):

    gt={}


    with open(annotation) as f:


        for line in f:


            d=line.strip().split(",")


            frame=int(d[0])

            obj_id=int(d[1])

            cls=int(d[7])


            if cls not in [1,2]:
                continue



            x=float(d[2])
            y=float(d[3])
            w=float(d[4])
            h=float(d[5])


            if frame not in gt:

                gt[frame]=[]


            gt[frame].append(
                [
                obj_id,
                x,
                y,
                x+w,
                y+h
                ]
            )



    return gt

In [59]:
def iou_matrix(gt,pred):

    if len(gt)==0 or len(pred)==0:

        return np.empty(
            (len(gt),len(pred))
        )


    gt_boxes=[]

    for g in gt:

        x1,y1,x2,y2 = g[1:]

        gt_boxes.append(
            [
                x1,
                y1,
                x2-x1,
                y2-y1
            ]
        )


    pred_boxes=[]

    for p in pred:

        x1,y1,x2,y2 = p[1:]

        pred_boxes.append(
            [
                x1,
                y1,
                x2-x1,
                y2-y1
            ]
        )


    return mm.distances.iou_matrix(
        gt_boxes,
        pred_boxes,
        max_iou=0.5
    )

In [60]:
def evaluate_tracking(
    detector,
    imgsz,
    tracker
):


    accumulator = (
        mm.MOTAccumulator(
            auto_id=True
        )
    )


    total_frames=0

    total_time=0



    for seq in sequences:


        frames=sorted(
            seq.glob("*.jpg")
        )


        gt=load_gt(
            ANN_DIR /
            f"{seq.name}.txt"
        )



        # reset tracker each sequence
        detector.predictor=None



        for img_path in tqdm(frames):


            frame_id=int(
                img_path.stem
            )


            img=cv2.imread(
                str(img_path)
            )


            start=time.time()



            result=detector.track(

                img,

                imgsz=imgsz,

                conf=0.2,

                classes=[0],

                tracker=tracker,

                persist=True,

                device=0,

                verbose=False

            )[0]


            torch.cuda.synchronize()


            total_time += (
                time.time()-start
            )


            preds=[]


            if result.boxes.id is not None:


                ids=(
                    result.boxes.id
                    .cpu()
                    .numpy()
                )


                boxes=(
                    result.boxes.xyxy
                    .cpu()
                    .numpy()
                )


                for i,b in zip(ids,boxes):


                    preds.append(
                        [
                        int(i),
                        b[0],
                        b[1],
                        b[2],
                        b[3]
                        ]
                    )



            gt_frame = gt.get(
                frame_id,
                []
            )



            accumulator.update(

                [g[0] for g in gt_frame],

                [p[0] for p in preds],

                iou_matrix(
                    gt_frame,
                    preds
                )

            )


            total_frames+=1



    fps = (
        total_frames/
        total_time
    )



    mh=mm.metrics.create()


    summary=mh.compute(

        accumulator,

        metrics=[
            "mota",
            "idf1",
            "num_switches",
            "mostly_tracked",
            "mostly_lost"
        ],

        name="result"

    )


    return {

        "FPS":fps,

        "MOTA":
        summary["mota"][0],


        "IDF1":
        summary["idf1"][0],


        "ID Switches":
        summary["num_switches"][0],


        "MT":
        summary["mostly_tracked"][0],


        "ML":
        summary["mostly_lost"][0]

    }

In [61]:
results=[]


for exp in experiments:


    for tracker_name,tracker_file in trackers.items():


        print(
            "\nRunning:",
            exp["name"],
            tracker_name
        )


        r=evaluate_tracking(

            exp["model"],

            exp["imgsz"],

            tracker_file

        )


        results.append(

        {

        "Detector":
        exp["name"],

        "Tracker":
        tracker_name,

        "FPS":
        round(r["FPS"],2),

        "MOTA":
        round(r["MOTA"],3),

        "IDF1":
        round(r["IDF1"],3),

        "ID Switches":
        r["ID Switches"],

        "Mostly Tracked":
        r["MT"],

        "Mostly Lost":
        r["ML"]

        }

        )



df=pd.DataFrame(results)

df


Running: YOLO11s_1280 ByteTrack


100%|██████████| 349/349 [00:16<00:00, 20.60it/s]
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:179: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  summary["mota"][0],
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:183: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  summary["idf1"][0],
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:187: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by pos


Running: YOLO11s_1280 BoT-SORT-No-GMC


100%|██████████| 349/349 [00:18<00:00, 19.20it/s]
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:179: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  summary["mota"][0],
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:183: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  summary["idf1"][0],
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:187: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by pos


Running: YOLO11s_1280 BoT-SORT-GMC


...


Running: RTDETR_960 BoT-SORT-No-GMC


100%|██████████| 349/349 [00:33<00:00, 10.54it/s]
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:179: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  summary["mota"][0],
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:183: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  summary["idf1"][0],
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:187: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by pos


Running: RTDETR_960 BoT-SORT-GMC


100%|██████████| 349/349 [02:20<00:00,  2.48it/s]
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:179: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  summary["mota"][0],
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:183: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  summary["idf1"][0],
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\3435656188.py:187: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by pos

,Detector,Tracker,FPS,MOTA,IDF1,ID Switches,Mostly Tracked,Mostly Lost
0,YOLO11s_1280,ByteTrack,36.56,0.270,0.376,78,9,87
1,YOLO11s_1280,BoT-SORT-No-GMC,32.25,0.179,0.298,33,5,103
2,YOLO11s_1280,BoT-SORT-GMC,4.18,0.173,0.293,12,7,102
3,YOLO11m_1280,ByteTrack,22.51,0.306,0.405,140,11,83
4,YOLO11m_1280,BoT-SORT-No-GMC,21.35,0.199,0.339,52,7,99
5,YOLO11m_1280,BoT-SORT-GMC,4.06,0.202,0.336,17,8,100
6,YOLO11l_1280,ByteTrack,18.66,0.349,0.445,131,15,81
7,YOLO11l_1280,BoT-SORT-No-GMC,18.24,0.241,0.380,36,8,96
8,YOLO11l_1280,BoT-SORT-GMC,4.02,0.239,0.383,13,9,94
9,RTDETR_960,ByteTrack,14.12,0.299,0.421,166,9,76


In [20]:
from pathlib import Path
import ultralytics

tracker_path = (
    Path(ultralytics.__file__).parent
    / "cfg"
    / "trackers"
)

print(tracker_path)

print(list(tracker_path.iterdir()))

c:\Users\VAMSEEKRISHNA.P\miniconda3\envs\aerial\lib\site-packages\ultralytics\cfg\trackers
[WindowsPath('c:/Users/VAMSEEKRISHNA.P/miniconda3/envs/aerial/lib/site-packages/ultralytics/cfg/trackers/botsort.yaml'), WindowsPath('c:/Users/VAMSEEKRISHNA.P/miniconda3/envs/aerial/lib/site-packages/ultralytics/cfg/trackers/bytetrack.yaml')]


In [23]:
import shutil


src = tracker_path/"bytetrack.yaml"


dst = "bytetrack_drone.yaml"


shutil.copy(
    src,
    dst
)


print("created")

created


In [44]:
SEQ_PATH = Path(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\VisDrone2019-MOT-val\VisDrone2019-MOT-val\sequences\uav0000086_00000_v"
)


frames = sorted(
    SEQ_PATH.glob("*.jpg")
)


print(len(frames))
model = YOLO("yolo11s.pt")

model.to("cuda")

464


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_runnin

In [45]:
def run_bytetrack_video():

    first=cv2.imread(str(frames[0]))

    h,w,_=first.shape


    writer=cv2.VideoWriter(

        "YOLO11s_ByteTrack.mp4",

        cv2.VideoWriter_fourcc(*"mp4v"),

        30,

        (w,h)

    )


    trajectories=defaultdict(list)


    times=[]



    for img_path in frames:


        frame=cv2.imread(
            str(img_path)
        )


        start=time.time()


        result=model.track(

            frame,

            imgsz=1280,

            conf=0.2,

            classes=[0],

            tracker="bytetrack.yaml",

            persist=True,

            device=0,

            verbose=False

        )[0]


        times.append(
            time.time()-start
        )



        if result.boxes.id is not None:


            ids=result.boxes.id.cpu().numpy()

            boxes=result.boxes.xyxy.cpu().numpy()



            for tid,box in zip(ids,boxes):


                x1,y1,x2,y2=map(
                    int,box
                )


                cx=(x1+x2)//2
                cy=(y1+y2)//2


                trajectories[int(tid)].append(
                    (cx,cy)
                )


                trajectories[int(tid)] = (
                    trajectories[int(tid)][-30:]
                )



                cv2.rectangle(
                    frame,
                    (x1,y1),
                    (x2,y2),
                    (0,255,0),
                    2
                )


                cv2.putText(
                    frame,
                    f"ID:{int(tid)}",
                    (x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0,255,0),
                    2
                )


                pts=np.array(
                    trajectories[int(tid)],
                    np.int32
                )


                cv2.polylines(
                    frame,
                    [pts],
                    False,
                    (0,0,255),
                    2
                )



        writer.write(frame)


    writer.release()


    print(
        "FPS:",
        len(frames)/sum(times)
    )

In [46]:
run_bytetrack_video()

FPS: 29.731830539614688


In [47]:
class CameraMotion:


    def __init__(self):

        self.prev=None

        self.orb=cv2.ORB_create(
            1000
        )


    def estimate(self,frame):


        gray=cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )


        if self.prev is None:

            self.prev=gray

            return np.eye(3)



        kp1,des1=self.orb.detectAndCompute(
            self.prev,None
        )


        kp2,des2=self.orb.detectAndCompute(
            gray,None
        )



        H=np.eye(3)


        if des1 is not None and des2 is not None:


            matcher=cv2.BFMatcher(
                cv2.NORM_HAMMING
            )


            matches=matcher.match(
                des1,des2
            )


            matches=sorted(
                matches,
                key=lambda x:x.distance
            )[:100]



            if len(matches)>10:


                src=np.float32(
                    [
                    kp1[m.queryIdx].pt
                    for m in matches
                    ]
                )


                dst=np.float32(
                    [
                    kp2[m.trainIdx].pt
                    for m in matches
                    ]
                )


                H,_=cv2.findHomography(
                    src,
                    dst,
                    cv2.RANSAC
                )


                if H is None:

                    H=np.eye(3)



        self.prev=gray


        return H

In [52]:
import time

def run_motion_comp_video():

    first = cv2.imread(
        str(frames[0])
    )

    h,w,_ = first.shape


    writer = cv2.VideoWriter(

        "YOLO11s_ByteTrack_GMC.mp4",

        cv2.VideoWriter_fourcc(*"mp4v"),

        30,

        (w,h)
    )


    trajectories = defaultdict(list)

    motion = CameraMotion()


    total_time = 0
    frame_count = 0


    for img_path in frames:


        frame = cv2.imread(str(img_path))


        start = time.time()


        # ---------- Camera Motion Compensation ----------
        if frame_count % 5 == 0:

            H = motion.estimate(frame)

        else:

            H = last_H


        last_H = H


        # ---------- YOLO + ByteTrack ----------
        result = model.track(

            frame,

            imgsz=1280,

            conf=0.2,

            classes=[0],

            tracker="bytetrack.yaml",

            persist=True,

            device=0,

            verbose=False

        )[0]


        if result.boxes.id is not None:


            ids = result.boxes.id.cpu().numpy()

            boxes = result.boxes.xyxy.cpu().numpy()


            for tid,box in zip(ids,boxes):


                x1,y1,x2,y2 = map(int,box)


                center = np.array(
                    [
                        [
                        (x1+x2)/2,
                        (y1+y2)/2,
                        1
                        ]
                    ]
                ).T


                warped = H @ center

                warped /= warped[2]


                cx = int(warped[0])
                cy = int(warped[1])


                trajectories[int(tid)].append(
                    (cx,cy)
                )


                trajectories[int(tid)] = (
                    trajectories[int(tid)][-30:]
                )


                cv2.rectangle(
                    frame,
                    (x1,y1),
                    (x2,y2),
                    (255,0,0),
                    2
                )


                cv2.putText(
                    frame,
                    f"GMC-ID:{int(tid)}",
                    (x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255,0,0),
                    2
                )


                pts = np.array(
                    trajectories[int(tid)],
                    np.int32
                )


                cv2.polylines(
                    frame,
                    [pts],
                    False,
                    (0,0,255),
                    2
                )


        # -------- FPS calculation ---------

        end = time.time()

        frame_time = end-start

        total_time += frame_time

        frame_count += 1


        current_fps = 1/frame_time


        # write FPS on video

        cv2.putText(

            frame,

            f"FPS: {current_fps:.2f}",

            (30,50),

            cv2.FONT_HERSHEY_SIMPLEX,

            1,

            (0,255,255),

            2

        )


        writer.write(frame)



    writer.release()


    avg_fps = frame_count / total_time


    print(
        "Average FPS:",
        round(avg_fps,2)
    )

In [53]:
run_motion_comp_video()

C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\882672301.py:107: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  cx = int(warped[0])
C:\Users\VAMSEEKRISHNA.P\AppData\Local\Temp\ipykernel_11136\882672301.py:108: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  cy = int(warped[1])


Average FPS: 23.89
